In [5]:
import csv
import random
import math
from google.colab import files

# ============================================================
# UPLOAD CSV FILE
# ============================================================

print("Please upload your CSV file:")
uploaded = files.upload()

filename = list(uploaded.keys())[0]

print("\nFile uploaded:", filename)


# ============================================================
# LOAD CSV FILE
# ============================================================

def loadcsv(filename):
    with open(filename, "r") as file:
        lines = csv.reader(file)

        # Skip header row
        next(lines)

        dataset = list(lines)

        for i in range(len(dataset)):
            dataset[i] = [float(x) for x in dataset[i]]

    return dataset


# ============================================================
# SPLIT DATASET INTO TRAINING AND TESTING DATA
# ============================================================

def splitdataset(dataset, splitratio):

    trainsize = int(len(dataset) * splitratio)

    trainset = []
    copy = list(dataset)

    while len(trainset) < trainsize:

        index = random.randrange(len(copy))

        trainset.append(copy.pop(index))

    return [trainset, copy]


# ============================================================
# SEPARATE DATASET ACCORDING TO CLASS
# ============================================================

def separatebyclass(dataset):

    separated = {}

    for i in range(len(dataset)):

        vector = dataset[i]

        if vector[-1] not in separated:
            separated[vector[-1]] = []

        separated[vector[-1]].append(vector)

    return separated


# ============================================================
# CALCULATE MEAN
# ============================================================

def mean(numbers):

    return sum(numbers) / float(len(numbers))


# ============================================================
# CALCULATE STANDARD DEVIATION
# ============================================================

def stdev(numbers):

    avg = mean(numbers)

    variance = sum(
        [pow(x - avg, 2) for x in numbers]
    ) / float(len(numbers) - 1)

    return math.sqrt(variance)


# ============================================================
# CALCULATE MEAN AND STANDARD DEVIATION
# ============================================================

def summarize(dataset):

    summaries = [
        (mean(attribute), stdev(attribute))
        for attribute in zip(*dataset)
    ]

    # Remove class column
    del summaries[-1]

    return summaries


# ============================================================
# CALCULATE SUMMARIES FOR EACH CLASS
# ============================================================

def summarizebyclass(dataset):

    separated = separatebyclass(dataset)

    summaries = {}

    for classvalue, instances in separated.items():

        summaries[classvalue] = summarize(instances)

    return summaries


# ============================================================
# CALCULATE GAUSSIAN PROBABILITY
# ============================================================

def calculateprobability(x, mean, stdev):

    exponent = math.exp(
        -(math.pow(x - mean, 2) /
          (2 * math.pow(stdev, 2)))
    )

    return (
        (1 / (math.sqrt(2 * math.pi) * stdev))
        * exponent
    )


# ============================================================
# CALCULATE PROBABILITY FOR EACH CLASS
# ============================================================

def calculateclassprobabilities(summaries, inputvector):

    probabilities = {}

    for classvalue, classsummaries in summaries.items():

        probabilities[classvalue] = 1

        for i in range(len(classsummaries)):

            mean_value, stdev_value = classsummaries[i]

            x = inputvector[i]

            probabilities[classvalue] *= calculateprobability(
                x,
                mean_value,
                stdev_value
            )

    return probabilities


# ============================================================
# PREDICT CLASS
# ============================================================

def predict(summaries, inputvector):

    probabilities = calculateclassprobabilities(
        summaries,
        inputvector
    )

    bestLabel = None
    bestProb = -1

    for classvalue, probability in probabilities.items():

        if bestLabel is None or probability > bestProb:

            bestProb = probability
            bestLabel = classvalue

    return bestLabel


# ============================================================
# GET PREDICTIONS
# ============================================================

def getpredictions(summaries, testset):

    predictions = []

    for i in range(len(testset)):

        result = predict(
            summaries,
            testset[i]
        )

        predictions.append(result)

    return predictions


# ============================================================
# CALCULATE ACCURACY
# ============================================================

def getaccuracy(testset, predictions):

    correct = 0

    for i in range(len(testset)):

        if testset[i][-1] == predictions[i]:

            correct += 1

    return (
        correct / float(len(testset))
    ) * 100.0


# ============================================================
# MAIN FUNCTION
# ============================================================

def main():

    # Make result reproducible
    random.seed(42)

    # Load dataset
    dataset = loadcsv(filename)

    print("\nTotal rows in dataset:", len(dataset))

    # 67% training and 33% testing
    splitratio = 0.67

    trainingset, testset = splitdataset(
        dataset,
        splitratio
    )

    print(
        "Split {0} rows into train={1} and test={2} rows"
        .format(
            len(dataset),
            len(trainingset),
            len(testset)
        )
    )

    # Prepare model
    summaries = summarizebyclass(trainingset)

    # Predict test data
    predictions = getpredictions(
        summaries,
        testset
    )

    # Calculate accuracy
    accuracy = getaccuracy(
        testset,
        predictions
    )

    print(
        "\nAccuracy of the classifier is : {0:.2f}%"
        .format(accuracy)
    )


# ============================================================
# RUN PROGRAM
# ============================================================

if __name__ == "__main__":
    main()

Please upload your CSV file:


Saving Naive-Bayes-Classification-Data.csv to Naive-Bayes-Classification-Data.csv

File uploaded: Naive-Bayes-Classification-Data.csv

Total rows in dataset: 12
Split 12 rows into train=8 and test=4 rows

Accuracy of the classifier is : 100.00%
